In [2]:
!pip install langchain-mcp-adapters

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-mcp-adapters]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-mcp 0.2.1 requires langchain-core~=0.3.37, but you have langchain-core 1.2.13 which is incompatible.


In [1]:
# Standard library imports
import asyncio
# Third-party imports for MCP (Model Context Protocol) and LangGraph
from langchain_mcp_adapters.client import MultiServerMCPClient # Connects to MCP servers
from langgraph.prebuilt import create_react_agent # Creates ReAct-style agents
from langgraph.checkpoint.memory import InMemorySaver # Provides conversation memory
from langchain_groq import ChatGroq 


/Users/anirudh/.local/share/virtualenvs/Desktop-sFnGVMJ4/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os,getpass
def _set_if_undefined(var:str):
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)
_set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [6]:
async def main():
    """
    Main function that sets up and runs an AI agent with access to multiple MCP servers.
    The agent can access Context7 library documentation and Met Museum collections.
    """
    client = MultiServerMCPClient(
        {
            "context7":{
                "url": "https://mcp.context7.com/mcp",
                "transport" : "streamable_http"
            },
            "met-museum":{
                "command": "npx",
                "args":["-y" ,"metmuseum-mcp"],
                "transport": "stdio"
            }
        }
    )
    groq_model = ChatGroq(
        model = "llama-3.3-70b-versatile",
        api_key = os.environ["GROQ_API_KEY"],
        temperature = 0
    )
    tools =  await client.get_tools()
    # Set up conversation memory using InMemorySaver
    # This allows the agent to remember previous messages in the conversation
    checkpointer = InMemorySaver()
    #Configuration for conversation persistence
    # The thread_id ensures all messages in this session are grouped together
    config = {"configurable": {"thread_id": "conversation_id"}}
    # Create the ReAct agent with all components
    # ReAct = Reasoning + Acting (agent can reason about and use tools)
    agent = create_react_agent(
        model = groq_model,
        tools = tools,
        checkpointer = checkpointer
    )
    # Send initial message to introduce the agent and its capabilities
    response = await agent.ainvoke(
        {"messages": [
            # System message defines the agent's role and personality
            {"role": "system", "content": "You are a smart, useful agent with tools to access code library documentation and the Met Museum collection."},
            # User message requests the agent to introduce itself
            {"role": "user", "content": "Give a brief introduction of what you do and the tools you can access."},
        ]},
        config=config  # Use the conversation thread for memory persistence
    )
    # Print the agent's response (last message in the conversation)
    print(response['messages'][-1].content)
    # Main interaction loop - allows continuous conversation with the agent
    while True:
        # Display menu options to the user
        choice = input("""
Menu:
1. Ask the agent a question
2. Quit
Enter your choice (1 or 2): """)
        if choice == "1":
            # Get user's question
            print("Your question")
            query = input("> ")
            # Send the user's question to the agent
            # The agent will have access to the full conversation history
            response = await agent.ainvoke(
                {"messages": query},        # User's current question
                config=config              # Maintains conversation thread
            )
            # Display the agent's response
            print(response['messages'][-1].content)
        else:
            # Exit the program for any choice other than "1"
            print("Goodbye!")
            break

await main()
if __name__ == "__main__":
    asyncio.run(main())

/var/folders/kv/9b0l8lv56tsc3yvfy9l9nq180000gn/T/ipykernel_15229/1885748397.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


I'm a smart and useful agent designed to assist with tasks related to programming and the Metropolitan Museum of Art (Met Museum) collection. I have access to various tools that enable me to provide helpful information and answers.

For programming-related queries, I can utilize the "resolve-library-id" tool to find the most relevant library ID based on a given query, and then use the "query-docs" tool to retrieve up-to-date documentation and code examples from Context7 for the specified library.

Regarding the Met Museum collection, I can use the "list-departments" tool to list all departments in the museum, the "search-museum-objects" tool to search for objects by title or other criteria, and the "get-museum-object" tool to retrieve information about a specific object by its ID.

I'm here to help with any questions or tasks you may have, so please feel free to ask!



Menu:
1. Ask the agent a question
2. Quit
Enter your choice (1 or 2):  1


Your question


>  What is the oldest art piece in your museum?.


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=list-departments{"__intent": "To find the oldest art piece in the museum, first need to list all departments."}</function>\n'}}